In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

c:\Users\lenovo\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
# Load the dataset
df=pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

In [12]:
numerical_features = ['annual_income', 'number_of_courses_viewed', 'interaction_count','lead_score']
categorical_features = ['lead_source', 'industry', 'employment_status', 'location']

In [17]:
#filling missing values for numerical features with 0
for feature in numerical_features:
    df[feature] = df[feature].fillna(0.0)

In [16]:
#filling missing values for categorical features with 'NA'
for feature in categorical_features:
    df[feature] = df[feature].fillna('NA')

In [18]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [21]:
#most frequent value for industry
df['industry'].value_counts().idxmax()

'technology'

In [22]:
#correlation matrix for numerical features
correlation_matrix = df[numerical_features].corr()

In [25]:
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [26]:
# Unstacking the correlation matrix into a Series of pairs
s = correlation_matrix.unstack()

# Remove self correlations (where index equals column)
s = s[s.index.get_level_values(0) != s.index.get_level_values(1)]

# Getting the maximum value and the corresponding pair
max_val = s.max()
max_pair = s.idxmax()

print(f'Max correlation: {max_val} between {max_pair}')


Max correlation: 0.9157457980418697 between ('interaction_count', 'lead_score')


In [27]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [29]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [31]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [32]:
from sklearn.metrics import mutual_info_score

In [35]:
mutual_info_score(df_train['lead_source'], y_train)

np.float64(0.03434605791617973)

In [36]:
#calculate mutual information for categorical features
def calculate_mutual_info(df, categorical_features, target):
    mutual_info = {}
    for feature in categorical_features:
        mi = mutual_info_score(df[feature], target)
        mutual_info[feature] = mi
    return mutual_info

In [37]:
mutual_info = calculate_mutual_info(df_train, categorical_features, y_train)

In [ ]:
#maximum mutual information and the corresponding feature
max_pair = max(mutual_info.items(), key=lambda item: item[1])
print(max_pair)  


('lead_source', np.float64(0.03434605791617973))


In [43]:
from sklearn.feature_extraction import DictVectorizer

In [44]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_features + numerical_features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_features + numerical_features].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
#train the logistic regression model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [46]:
#fit the model on the training data
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [49]:
prediction = model.predict(X_val)

In [56]:
#accuracy of the model on the validation set
from sklearn.metrics import accuracy_score

accuracy_score(y_val, prediction)

0.645

In [55]:
X_train

array([[6.5980e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       [7.1662e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        0.0000e+00],
       [4.0220e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [7.1532e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [4.8836e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [6.8969e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00]])

In [57]:
#function to to do ohe and train the model
def train_and_score(features):
    # One-hot encode using only the given features
    dv = DictVectorizer(sparse=False)

    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    # Train and evaluate
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [58]:
all_features = categorical_features + numerical_features

In [59]:
# 1. Baseline: all features
base_acc = train_and_score(all_features)
print("Baseline accuracy:", base_acc)

# 2. Drop one feature at a time
scores = {}
for feature in all_features:
    remaining = [f for f in all_features if f != feature]
    scores[feature] = train_and_score(remaining)

# 3. How much did accuracy drop without each feature?
diffs = {f: base_acc - acc for f, acc in scores.items()}

Baseline accuracy: 0.645


In [60]:
diffs

{'lead_source': 0.0030000000000000027,
 'industry': 0.0,
 'employment_status': 0.0,
 'location': 0.0,
 'annual_income': -0.07899999999999996,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004,
 'lead_score': 0.0010000000000000009}

In [61]:
for f, d in sorted(diffs.items(), key=lambda x: x[1], reverse=True):
    print(f"{f:25s} accuracy without it: {scores[f]:.4f}   drop: {d:+.4f}")

interaction_count         accuracy without it: 0.6010   drop: +0.0440
lead_source               accuracy without it: 0.6420   drop: +0.0030
number_of_courses_viewed  accuracy without it: 0.6430   drop: +0.0020
lead_score                accuracy without it: 0.6440   drop: +0.0010
industry                  accuracy without it: 0.6450   drop: +0.0000
employment_status         accuracy without it: 0.6450   drop: +0.0000
location                  accuracy without it: 0.6450   drop: +0.0000
annual_income             accuracy without it: 0.7240   drop: -0.0790


In [64]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_features + numerical_features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_features + numerical_features].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [65]:
#changing the regularization parameter C
c=[0.000001, 0.00001, 0.0001, 0.001]
accuracy_scores = {}
for c_value in c:
    model=LogisticRegression(solver='liblinear', C=c_value, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    prediction = model.predict(X_val)
    accuracy_scores[c_value] = accuracy_score(y_val, prediction)

In [66]:
accuracy_scores

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}